# Task 1: Text Classification Benchmark

This notebook benchmarks multiple text classification algorithms (Classifiers) and feature extractors (Vectorizers) on the **20 Newsgroups** dataset.

**Classifiers**
- Multinomial Naive Bayes
- Logistic Regression
- Support Vector Machine (LinearSVC)
- Decision Tree
- GridSearchCV

**Vectorizers**
- CountVectorizer
- TfidfVectorizer
- Word2Vec 
- Doc2Vec 
- Fasttext
  

## Import Libraries


In [32]:
from pprint import pprint
from time import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin, clone

from gensim.models import Word2Vec, Doc2Vec, FastText
from gensim.models.doc2vec import TaggedDocument

## Load Data


In [33]:
categories = [
    'alt.atheism',
    'talk.religion.misc',
]

print("Loading 20 newsgroups dataset for categories:")
print(categories)

train = fetch_20newsgroups(subset='train', categories=categories,
                           remove=('headers', 'footers', 'quotes'))
test  = fetch_20newsgroups(subset='test',  categories=categories,
                           remove=('headers', 'footers', 'quotes'))

X_train, y_train = train.data, train.target
X_test,  y_test  = test.data,  test.target

print(f"Train: {len(X_train)} documents | {len(train.target_names)} categories")
print(f"Test:  {len(X_test)}  documents")

Loading 20 newsgroups dataset for categories:
['alt.atheism', 'talk.religion.misc']
Train: 857 documents | 2 categories
Test:  570  documents


## Custom Transformers


In [34]:
class Word2VecTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, min_count=1, epochs=10):
        self.vector_size = vector_size
        self.window      = window
        self.min_count   = min_count
        self.epochs      = epochs

    def fit(self, X, y=None):
        tokenized = [doc.split() for doc in X]
        self.model_ = Word2Vec(tokenized, vector_size=self.vector_size,
                               window=self.window, min_count=self.min_count,
                               epochs=self.epochs)
        return self

    def transform(self, X):
        tokenized = [doc.split() for doc in X]
        return np.array([
            np.mean([self.model_.wv[w] for w in words if w in self.model_.wv]
                    or [np.zeros(self.vector_size)], axis=0)
            for words in tokenized
        ])


class Doc2VecTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, min_count=1, epochs=10):
        self.vector_size = vector_size
        self.window      = window
        self.min_count   = min_count
        self.epochs      = epochs

    def fit(self, X, y=None):
        tagged = [TaggedDocument(doc.split(), [i]) for i, doc in enumerate(X)]
        self.model_ = Doc2Vec(tagged, vector_size=self.vector_size,
                              window=self.window, min_count=self.min_count,
                              epochs=self.epochs)
        return self

    def transform(self, X):
        return np.array([self.model_.infer_vector(doc.split()) for doc in X])


class FastTextTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, min_count=1, epochs=10):
        self.vector_size = vector_size
        self.window      = window
        self.min_count   = min_count
        self.epochs      = epochs

    def fit(self, X, y=None):
        tokenized = [doc.split() for doc in X]
        self.model_ = FastText(tokenized, vector_size=self.vector_size,
                               window=self.window, min_count=self.min_count,
                               epochs=self.epochs)
        return self

    def transform(self, X):
        tokenized = [doc.split() for doc in X]
        return np.array([
            np.mean([self.model_.wv[w] for w in words]
                    or [np.zeros(self.vector_size)], axis=0)
            for words in tokenized
        ])

## Define Vectorizers & Classifiers


In [35]:
sparse_vectorizers = {
    'CountVectorizer': CountVectorizer(),
    'TfidfVectorizer': TfidfVectorizer(),
}

dense_vectorizers = {
    'Word2Vec': Word2VecTransformer(),
    'Doc2Vec':  Doc2VecTransformer(),
    'FastText': FastTextTransformer(),
}

# MultinomialNB requires non-negative features, sparse only
sparse_classifiers = {
    'MultinomialNB':      MultinomialNB(),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'LinearSVC':          LinearSVC(max_iter=1000),
    'DecisionTree':       DecisionTreeClassifier(),
}

dense_classifiers = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'LinearSVC':          LinearSVC(max_iter=1000),
    'DecisionTree':       DecisionTreeClassifier(),
}


## Benchmark Loop


In [36]:
results = []

def run_benchmark(vec_name, clf_name, vectorizer, classifier):
    pipeline = Pipeline([
        ('vect', clone(vectorizer)),
        ('clf',  clone(classifier)),
    ])

    t0 = time()
    pipeline.fit(X_train, y_train)
    fit_time = time() - t0

    t0 = time()
    y_pred = pipeline.predict(X_test)
    pred_time = time() - t0

    acc = accuracy_score(y_test, y_pred)
    print(f"  {vec_name:<20} + {clf_name:<22} | acc={acc:.4f} | fit={fit_time:.1f}s | pred={pred_time:.3f}s")

    results.append({
        'vectorizer': vec_name,
        'classifier': clf_name,
        'accuracy':   round(acc, 4),
        'fit_time':   round(fit_time, 2),
        'pred_time':  round(pred_time, 4),
    })


print("\nSparse Vectorizers")
for vec_name, vectorizer in sparse_vectorizers.items():
    for clf_name, classifier in sparse_classifiers.items():
        run_benchmark(vec_name, clf_name, vectorizer, classifier)

print("\nDense Vectorizers")
for vec_name, vectorizer in dense_vectorizers.items():
    for clf_name, classifier in dense_classifiers.items():
        run_benchmark(vec_name, clf_name, vectorizer, classifier)


Sparse Vectorizers
  CountVectorizer      + MultinomialNB          | acc=0.7018 | fit=0.0s | pred=0.019s
  CountVectorizer      + LogisticRegression     | acc=0.6579 | fit=0.5s | pred=0.025s
  CountVectorizer      + LinearSVC              | acc=0.6456 | fit=0.1s | pred=0.024s
  CountVectorizer      + DecisionTree           | acc=0.5667 | fit=0.1s | pred=0.017s
  TfidfVectorizer      + MultinomialNB          | acc=0.5772 | fit=0.0s | pred=0.017s
  TfidfVectorizer      + LogisticRegression     | acc=0.6737 | fit=0.0s | pred=0.040s
  TfidfVectorizer      + LinearSVC              | acc=0.7105 | fit=0.1s | pred=0.044s
  TfidfVectorizer      + DecisionTree           | acc=0.5456 | fit=0.1s | pred=0.018s

Dense Vectorizers


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Word2Vec             + LogisticRegression     | acc=0.5544 | fit=0.6s | pred=0.071s


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Word2Vec             + LinearSVC              | acc=0.5772 | fit=0.7s | pred=0.045s
  Word2Vec             + DecisionTree           | acc=0.5281 | fit=0.6s | pred=0.046s


...

  Doc2Vec              + LogisticRegression     | acc=0.5789 | fit=1.3s | pred=0.610s
  Doc2Vec              + LinearSVC              | acc=0.5947 | fit=1.3s | pred=0.457s
  Doc2Vec              + DecisionTree           | acc=0.5351 | fit=1.3s | pred=0.444s


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  FastText             + LogisticRegression     | acc=0.5649 | fit=2.8s | pred=0.282s


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  FastText             + LinearSVC              | acc=0.5632 | fit=2.9s | pred=0.179s
  FastText             + DecisionTree           | acc=0.5737 | fit=2.9s | pred=0.191s


## GridSearch (TfidfVectorizer + LinearSVC)


In [29]:
print("\nGridSearch")
gs_pipeline = Pipeline([
    ('vect', TfidfVectorizer()),
    ('clf',  LinearSVC(max_iter=1000)),
])
parameters = {
    'vect__max_df':      (0.5, 0.75, 1.0),
    'vect__ngram_range': ((1, 1), (1, 2)),
    'clf__C':            (0.1, 1.0, 10.0),
    'clf__penalty':      ('l2',),
}
grid_search = GridSearchCV(gs_pipeline, parameters, cv=5, n_jobs=-1, verbose=1)
t0 = time()
grid_search.fit(X_train, y_train)
fit_time = time() - t0
t0 = time()
y_pred_gs = grid_search.predict(X_test)
pred_time = time() - t0
acc_gs = accuracy_score(y_test, y_pred_gs)
print(f"\nBest CV score: {grid_search.best_score_:.4f}")
print(f"Test accuracy: {acc_gs:.4f}")
print("Best parameters set:")
best_params = grid_search.best_estimator_.get_params()
for param in sorted(parameters.keys()):
    print(f"  {param}: {best_params[param]!r}")
results.append({
    'vectorizer': 'TfidfVectorizer',
    'classifier': 'GridSearch (LinearSVC)',
    'accuracy':   round(acc_gs, 4),
    'fit_time':   round(fit_time, 2),
    'pred_time':  round(pred_time, 4),
})



GridSearch
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Best CV score: 0.8005
Test accuracy: 0.6895
Best parameters set:
  clf__C: 10.0
  clf__penalty: 'l2'
  vect__max_df: 0.5
  vect__ngram_range: (1, 2)


## Results Table


In [30]:
df = pd.DataFrame(results).sort_values('accuracy', ascending=False)
b
print("\nFinal Results")
print(df.to_string(index=False))


Final Results
     vectorizer             classifier  accuracy  fit_time  pred_time
TfidfVectorizer              LinearSVC    0.7105      0.06     0.0282
CountVectorizer          MultinomialNB    0.7018      0.07     0.0202
TfidfVectorizer GridSearch (LinearSVC)    0.6895      3.41     0.0440
TfidfVectorizer     LogisticRegression    0.6737      0.04     0.0242
CountVectorizer     LogisticRegression    0.6579      0.18     0.0483
CountVectorizer              LinearSVC    0.6456      0.16     0.0334
        Doc2Vec              LinearSVC    0.6035      1.34     0.4483
        Doc2Vec     LogisticRegression    0.5807      1.32     0.5628
       Word2Vec              LinearSVC    0.5789      0.66     0.0465
TfidfVectorizer          MultinomialNB    0.5772      0.03     0.0174
       FastText           DecisionTree    0.5754      2.89     0.1810
       Word2Vec     LogisticRegression    0.5632      0.58     0.0638
CountVectorizer           DecisionTree    0.5614      0.08     0.0172
     

In [31]:
print(f"\nWinner: {df.iloc[0]['vectorizer']} + {df.iloc[0]['classifier']} (acc={df.iloc[0]['accuracy']:.4f})")


Winner: TfidfVectorizer + LinearSVC (acc=0.7105)
